# D-MTHD Wikipedia personal-attacks benchmark

Settings: Accelerator **GPU T4 x2**, Internet **On** (the corpus is downloaded from Figshare). No dataset needs attaching.

This benchmark is longer than the tweets one (69k comments at 256 tokens). Run it as two versions if needed: the first session trains and caches the teachers; if it stops before the students finish, attach that version's output as an input, set `RESUME_FROM` below to its path, and run again. Every finished run is skipped.

**Save Version -> Save & Run All (Commit)**.

In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        assert tree, "clone failed and no code found among the inputs"
        shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(tree[0]))), DEST)
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print("code ready")

In [ ]:
import os, subprocess, glob, sys
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd', TEACHER_EPOCHS='3', DISAGREEMENT='1')
env['RESUME_FROM'] = ''   # second version: attach the first version's output as an input and put its path here
raw = None   # Wikipedia is downloaded from Figshare by the driver
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'wikipedia', '--stage', 'all'] + (['--raw', raw] if raw else [])
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
!cd /kaggle/working && tar czf dmthd_wikipedia_runs.tgz runs cache/wikipedia/meta.json data/wikipedia/report.json && ls -la dmthd_wikipedia_runs.tgz
!python -m dmthd.aggregate --runs /kaggle/working/runs/wikipedia